# Constructing a RAG System and Enhancing it with HyDE

This notebook builds a baseline RAG workflow and improves retrieval quality with HyDE for question answering over a technical document.

## Recommended Hardware

This notebook can run on the following hardware or remote resources

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/rag/3.rag-HyDE.ipynb)  


## Software Environment

Install ROCm on your system

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Build a basic RAG pipeline using a real technical document
- Compare standard retrieval with HyDE-based retrieval
- Understand how prompt design affects retrieval quality

### Install Dependencies

Install the package dependencies needed for this notebook.

First, get the `aup_config.py` script locally with the package dependencies.

In [ ]:
!wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/ai-agents/rag/aup_config.py

Install the dependencies. This step may take a few minutes and only needs to be done once.

In [ ]:
from aup_config import aup_setup
aup_setup()

## Build the RAG Pipeline

This section explains how to configure and build the RAG pipeline

### Setup Indexing and the Query Engine
Import the necessary libraries

In [ ]:
import requests
import os
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

## RAG System Preparation

### Download Document

We will download a copy of the [Vitis HLS User Guide](https://docs.amd.com/r/en-US/ug1399-vitis-hls) as content for the RAG database.

In [ ]:
base_url = 'https://docs.amd.com/api/khub/maps/JQtJoZLV908LbR5xokDqLw/attachments/9sLLuMlUume6oVQ1~HLSdg-JQtJoZLV908LbR5xokDqLw/content?download=true&Ft-Calling-App=ft%2Fturnkey-portal&Ft-Calling-App-Version=5.2.44'
download_dir = 'data_hls'
pdf_filename = 'vitis_hls_ug.pdf'

os.makedirs(download_dir, exist_ok=True)
if not os.path.isfile(os.path.join(download_dir, pdf_filename)):
    response = requests.get(base_url, stream=True)
    if response.status_code == 200:
        pdf_path = os.path.join(download_dir, pdf_filename)
        with open(pdf_path, 'wb') as file:
            file.write(response.content)

In [ ]:
loader = PyPDFLoader(os.path.join(download_dir, pdf_filename), mode="page")
pdf_doc = loader.load()
print(f"Number of pages in the PDF: {len(pdf_doc)}")

Remove table of contents, and appendices 

In [ ]:
docs = pdf_doc[10:899].copy()
print(f"Number of pages after filtering: {len(docs)}")

Clean up headers and footers

In [ ]:
for doc in docs:
    doc.page_content = doc.page_content.replace('\nSend Feedback', '')
    doc.page_content = re.sub(r'^Vitis HLS User Guide\s*.*$', '',doc.page_content, flags=re.MULTILINE)
    doc.page_content = re.sub(r'^UG1399\s*.*$', '',doc.page_content, flags=re.MULTILINE)

Chunk the document

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=32)
documents_split = text_splitter.split_documents(docs)
print(f"Number of documents after splitting: {len(documents_split)}")

### Embeddings

Instantiate the embedding model that feeds the documents into the vector database

In [ ]:
base_url = "http://localhost:11434/v1/"
api_key = ""
embed_model = OpenAIEmbeddings(model="nomic-embed-text:v1.5", base_url=base_url, api_key=api_key, check_embedding_ctx_length=False)

### Vector Database

Instantiate a Chroma vector database with the document chunks

In [ ]:
vectordb = Chroma.from_documents(documents_split, embed_model)
retriever = vectordb.as_retriever()

## RAG System

In [ ]:
model = ChatOpenAI(model="qwen3:8b", base_url=base_url, api_key=api_key, temperature=0)

In [ ]:
template = """
Use the following pieces of context to answer the question at the end.

If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum and keep the answer as concise as possible.

{context}

Question: {question}?

Helpful Answer:
"""

custom_rag_prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} |
             custom_rag_prompt | model | StrOutputParser())

In [ ]:
query = "Interfaces Vitis HLS Support"

In [ ]:
rag_chain.invoke(query)

## RAG System + HyDE

### Hypothetical Document Generation

Generate a hypothetical answer document to improve retrieval for difficult queries

In [ ]:
template = """You are an expert on Vitis HLS.
Write a detailed document on the topic: '{user_query}'.

Your response should be concise and include all key points that would be found in the top search results.
Only focus on the topic, do not explain what Vitis HLS is unless is part of the user question.
"""

prompt_hyde = ChatPromptTemplate.from_template(template)

In [ ]:
generate_docs_for_retrieval = (
    prompt_hyde | model | StrOutputParser()
)

In [ ]:
generate_docs_for_retrieval.invoke({'user_query': query})

In [ ]:
retrieval_chain = generate_docs_for_retrieval | retriever

In [ ]:
retrieved_docs = retrieval_chain.invoke({'user_query': query})

In [ ]:
rag_hyde_chain = (
    custom_rag_prompt |
    model |
    StrOutputParser()
)

In [ ]:
rag_hyde_chain.invoke({"context": retrieved_docs, "question": query})

In [ ]:
rag_hyde_chain_full = (
    {"user_query": RunnablePassthrough()} |
    RunnablePassthrough.assign(hypothetical_doc=prompt_hyde | model | StrOutputParser()) |
    RunnablePassthrough.assign(context=lambda x: format_docs(retriever.invoke(x["hypothetical_doc"]))) |
    RunnablePassthrough.assign(question=lambda x: x["user_query"]) |
    custom_rag_prompt | 
    model | 
    StrOutputParser()
)

In [ ]:
rag_hyde_chain_full.invoke("How can pragmas be used in Vitis HLS?")

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://rocm.docs.amd.com/projects/ai-developer-hub/en/v3.1/notebooks/inference/rag_ollama_llamaindex.html">RAG System with LlamaIndex</a></li>
</ul>
</div>

## Conclusions

In this notebook, we explored a traditional RAG system and an enhanced RAG system with Hypothetical Document Embeddings

---

[AMD University Program](https://www.amd.com/aup)

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT